In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import tempfile

import cv2
import numpy as np
import torch

ROOT = Path("/Users/pishty/ws/vjepa2.1")
sys.path.insert(0, str(ROOT / "src"))

from vjepa2_latents.gradio_components.latent_source.config import CHECKPOINT_DIR, DEFAULT_MODEL_NAME, DEFAULT_VIDEO, VENDOR_REPO
from vjepa2_latents.gradio_components.latent_source.extractor import (
    auto_device,
    download_checkpoint_if_needed,
    load_encoder,
    preprocess_video,
    probe_video,
    read_video_frames,
    reshape_patch_tokens,
    run_encoder_synchronously,
    select_frame_indices,
)

model_name = DEFAULT_MODEL_NAME  # vit_base_384 = the 80M model
num_frames = 8
crop_size = (384, 384)
device = auto_device("auto")


def build_demo_video(video_path: Path, *, frame_count: int = 16, size: tuple[int, int] = (384, 384), fps: int = 24) -> Path:
    video_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, size)
    if not writer.isOpened():
        raise RuntimeError(f"Could not create demo video at {video_path}")
    width, height = size
    for index in range(frame_count):
        frame = np.zeros((height, width, 3), dtype=np.uint8)
        frame[..., 0] = np.linspace(0, 255, width, dtype=np.uint8)[None, :]
        frame[..., 1] = np.linspace(255, 0, height, dtype=np.uint8)[:, None]
        frame[..., 2] = (index * 17) % 255
        x0 = 32 + (index * 11) % max(width - 96, 1)
        y0 = 48 + (index * 7) % max(height - 96, 1)
        cv2.rectangle(frame, (x0, y0), (min(x0 + 64, width - 1), min(y0 + 64, height - 1)), (255, 255, 255), -1)
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    writer.release()
    return video_path


if DEFAULT_VIDEO.exists():
    video_path = DEFAULT_VIDEO
else:
    temp_dir = Path(tempfile.mkdtemp(prefix="vjepa2-demo-video-"))
    video_path = build_demo_video(temp_dir / "demo.mp4", frame_count=max(num_frames, 16), size=crop_size)

video_meta = probe_video(video_path)
frame_indices = select_frame_indices(
    video_fps=video_meta["fps"],
    frame_count=video_meta["frame_count"],
    num_frames=num_frames,
)
frames = read_video_frames(video_path, frame_indices)
video_tensor = preprocess_video(frames, crop_size)
checkpoint_path = download_checkpoint_if_needed(model_name, None, CHECKPOINT_DIR)
encoder = load_encoder(
    model_name=model_name,
    num_frames=num_frames,
    checkpoint_path=checkpoint_path,
    vendor_repo=VENDOR_REPO,
    device=device,
)


def run_once() -> dict[str, object]:
    raw_tokens, encoder_seconds = run_encoder_synchronously(encoder, video_tensor, device)
    time_patches = num_frames // 2
    height_patches = crop_size[0] // 16
    width_patches = crop_size[1] // 16
    latent_grid, tokens_stripped = reshape_patch_tokens(
        raw_tokens,
        time_patches=time_patches,
        height_patches=height_patches,
        width_patches=width_patches,
    )
    return {
        "raw_tokens": raw_tokens.detach().cpu(),
        "latent_grid": latent_grid.detach().cpu().numpy().astype(np.float32, copy=False),
        "encoder_seconds": encoder_seconds,
        "tokens_stripped": tokens_stripped,
    }


run_1 = run_once()
run_2 = run_once()
latent_1 = run_1["latent_grid"]
latent_2 = run_2["latent_grid"]

delta = latent_1 - latent_2
flat_1 = latent_1.reshape(-1, latent_1.shape[-1])
flat_2 = latent_2.reshape(-1, latent_2.shape[-1])
flat_1_norm = flat_1 / np.clip(np.linalg.norm(flat_1, axis=1, keepdims=True), 1e-12, None)
flat_2_norm = flat_2 / np.clip(np.linalg.norm(flat_2, axis=1, keepdims=True), 1e-12, None)
per_token_cosine = np.sum(flat_1_norm * flat_2_norm, axis=1).reshape(latent_1.shape[1:4])

print(f"Video: {video_path}")
print(f"Model: {model_name} | device: {device}")
print(f"Frames: {frame_indices}")
print(f"Latent shape: {latent_1.shape}")
print(f"Run 1 encoder seconds: {run_1['encoder_seconds']:.3f}")
print(f"Run 2 encoder seconds: {run_2['encoder_seconds']:.3f}")
print(f"Tokens stripped: {run_1['tokens_stripped']} / {run_2['tokens_stripped']}")
print(f"Allclose: {np.allclose(latent_1, latent_2, rtol=1e-5, atol=1e-6)}")
print(f"Max abs diff: {float(np.max(np.abs(delta))):.8f}")
print(f"Mean abs diff: {float(np.mean(np.abs(delta))):.8f}")
print(f"Mean token cosine similarity: {float(np.mean(per_token_cosine)):.8f}")
print(f"Min token cosine similarity: {float(np.min(per_token_cosine)):.8f}")
print(f"Max token cosine similarity: {float(np.max(per_token_cosine)):.8f}")

[vjepa2] Found cached checkpoint: /Users/pishty/ws/vjepa2.1/checkpoints/vjepa2_1_vitb_dist_vitG_384.pt
/Users/pishty/ws/vjepa2.1/.venv/lib/python3.13/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[vjepa2] Building encoder vit_base_384 for 8 frames on mps
/Users/pishty/ws/vjepa2.1/.venv/lib/python3.13/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[vjepa2] Building encoder vit_base_384 for 8 frames on mps
[vjepa2] Loading checkpoint weights from /Users/pishty/ws/vjepa2.1/checkpoints/vjepa2_1_vitb_dist_vitG_384.pt
[vjepa2] Loading checkpoint weights from /Users/pishty/ws/vjepa2.1/checkpoint

Video: /Users/pishty/ws/vjepa2.1/testvideo.mp4
Model: vit_base_384 | device: mps
Frames: [0, 1, 2, 3, 4, 5, 6, 7]
Latent shape: (1, 4, 24, 24, 768)
Run 1 encoder seconds: 0.497
Run 2 encoder seconds: 0.222
Tokens stripped: 0 / 0
Allclose: True
Max abs diff: 0.00000000
Mean abs diff: 0.00000000
Mean token cosine similarity: 1.00000000
Min token cosine similarity: 0.99999976
Max token cosine similarity: 1.00000024
